# 🏎️ F1 Data Loader
## Download Incrementale Dati F1 da FastF1 API

---

### Descrizione
Questo notebook gestisce il download dei dati telemetrici F1 dalla libreria open-source **FastF1**.  
Il download è **incrementale** per rispettare i rate limit dell'API (~100 richieste/ora per IP).

### Workflow Consigliato
Per ogni anno da scaricare:
1. Imposta `YEAR_TO_LOAD = <anno>`
2. Esegui tutte le celle
3. Scarica il file `f1_dataset_combined.pkl`
4. **Disconnetti il runtime** (per evitare ban IP)
5. Ricarica il file pickle e ripeti per l'anno successivo

### Output
| File | Descrizione |
|------|-------------|
| `f1_dataset_combined.pkl` | Dataset cumulativo (cresce ad ogni esecuzione) |
| `f1_cache/` | Cache FastF1 locale (riutilizzabile) |

---
## 1. Setup

In [ ]:
# Setup iniziale
!pip install fastf1 -q

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

import os
import pandas as pd
import numpy as np
import fastf1
from datetime import datetime

# Crea directory cache
os.makedirs('f1_cache', exist_ok=True)
fastf1.Cache.enable_cache('f1_cache')

In [ ]:
class F1DataLoaderIncremental:
    """
    Loader incrementale per dati F1.
    
    Caratteristiche:
    - Carica un anno alla volta per evitare rate limit API
    - Salva progressivamente su file pickle
    - Gestisce automaticamente duplicati
    - Include dati meteo quando disponibili
    """

    def __init__(self, cache_dir='f1_cache', output_file='f1_dataset_combined.pkl'):
        self.cache_dir = cache_dir
        self.output_file = output_file
        self.df = None
        fastf1.Cache.enable_cache(cache_dir)

    def load_existing_data(self):
        """Carica dataset esistente se presente."""
        if os.path.exists(self.output_file):
            print(f"Trovati dati esistenti: {self.output_file}")
            existing = pd.read_pickle(self.output_file)
            years_present = sorted(existing['Year'].unique())
            print(f"   Anni già caricati: {years_present}")
            return existing
        else:
            print("Nessun dato esistente - prima esecuzione")
            return None

    def load_single_year(self, year, max_rounds=None):
        """Carica dati di UN SOLO anno."""
        all_laps = []
        errors = []

        schedule = fastf1.get_event_schedule(year)
        races = schedule[schedule['EventFormat'] != 'testing']
        n_rounds = len(races) if max_rounds is None else min(max_rounds, len(races))

        print(f"🏁 Gare da caricare: {n_rounds}\n")

        successful = 0

        for idx in range(n_rounds):
            race = races.iloc[idx]
            round_num = race['RoundNumber']
            race_name = race['EventName']
            location = race['Location']

            session = fastf1.get_session(year, round_num, 'R')
            session.load(laps=True, telemetry=False, weather=True, messages=False)

            laps = session.laps.copy()

            # Filtra giri validi
            laps = laps[
                (laps['IsAccurate'] == True) &
                (laps['LapTime'].notna()) &
                (laps['Deleted'].isna())
            ].copy()

            if len(laps) > 0:
                # Metadati gara
                laps['Year'] = year
                laps['Round'] = round_num
                laps['RaceName'] = race_name
                laps['Circuit'] = location
                laps['Country'] = race['Country']
                laps['EventDate'] = race['EventDate']

                # Meteo
                if hasattr(session, 'weather_data') and session.weather_data is not None:
                    weather = session.weather_data
                    if not weather.empty:
                        laps['AirTemp'] = weather['AirTemp'].mean()
                        laps['TrackTemp'] = weather['TrackTemp'].mean()
                        laps['Humidity'] = weather['Humidity'].mean()
                        laps['Pressure'] = weather['Pressure'].mean()
                        laps['WindSpeed'] = weather['WindSpeed'].mean()
                        laps['Rainfall'] = weather['Rainfall'].any()

                laps = self._add_gap_features(laps)
                all_laps.append(laps)
                successful += 1

        if all_laps:
            df_year = pd.concat(all_laps, ignore_index=True)

            print(f"\n{'='*60}")
            print(f"📊 RIEPILOGO ANNO {year}")
            print(f"{'='*60}")
            print(f"Gare caricate:  {len(df_year.groupby('Round'))}")
            print(f"Giri totali:    {len(df_year):,}")
            print(f"Piloti:         {df_year['Driver'].nunique()}")
            print(f"Team:           {df_year['Team'].nunique()}")
            print(f"Circuiti:       {df_year['Circuit'].nunique()}")

            return df_year
        else:
            print(f"❌ Nessun dato caricato per {year}")
            return None

    def _add_gap_features(self, laps):
        """Aggiungi gap dal leader e stint info."""
        laps = laps.copy()
        laps = laps.sort_values(['LapNumber', 'Position']).reset_index(drop=True)

        gap_leader = []
        for lap_num in laps['LapNumber'].unique():
            lap_mask = laps['LapNumber'] == lap_num
            lap_data = laps[lap_mask]

            if len(lap_data) > 0 and 'Time' in lap_data.columns:
                leader_time = lap_data.iloc[0]['Time']
                gaps = (lap_data['Time'] - leader_time).dt.total_seconds()
                gap_leader.extend(gaps.tolist())
            else:
                gap_leader.extend([0.0] * len(lap_data))

        laps['GapToLeader'] = gap_leader

        if 'Stint' in laps.columns:
            laps['StintProgress'] = laps.groupby(['Driver', 'Stint']).cumcount() + 1

        return laps

    def combine_and_save(self, new_data, existing_data=None):
        """Combina e salva dataset."""
        if existing_data is not None:
            combined = pd.concat([existing_data, new_data], ignore_index=True)
            combined = combined.drop_duplicates(
                subset=['Year', 'Round', 'Driver', 'LapNumber'],
                keep='first'
            )
        else:
            combined = new_data

        combined.to_pickle(self.output_file)
        print(f"\n💾 SALVATO: {self.output_file}")

        return combined

    def load_year_incremental(self, year, max_rounds=None, force_reload=False):
        """Carica un anno e lo aggiunge al dataset."""
        print("\n")
        print("="*60)
        print(f"Anno da caricare: {year}")
        print(f"File output: {self.output_file}")
        print(f"Force reload: {force_reload}")

        existing = self.load_existing_data()

        if not force_reload and existing is not None and year in existing['Year'].values:
            print(f"⚠️  Anno {year} già presente!")
            print(f"   Usa force_reload=True per ricaricare\n")
            return existing

        if force_reload and existing is not None and year in existing['Year'].values:
            print(f"🔄 Rimozione dati esistenti per {year}...")
            existing = existing[existing['Year'] != year]
            print(f"   Rimossi dati del {year}\n")

        print("="*60)
        new_data = self.load_single_year(year, max_rounds)

        if new_data is None:
            print(f"❌ Impossibile caricare anno {year}")
            return existing if existing is not None else None

        combined = self.combine_and_save(new_data, existing)

        return combined

---
## 2. Configurazione

⚠️ **Modifica `YEAR_TO_LOAD` per ogni esecuzione!**

In [ ]:
YEAR_TO_LOAD = 2024  # ⬅️ MODIFICA QUI

MAX_ROUNDS = None    # None = tutte le gare
FORCE_RELOAD = False # True per ricaricare anno già presente

---
## 3. Esecuzione

In [ ]:
loader = F1DataLoaderIncremental(
    cache_dir='f1_cache',
    output_file='f1_dataset_combined.pkl'
)

df_f1 = loader.load_year_incremental(
    year=YEAR_TO_LOAD,
    max_rounds=MAX_ROUNDS,
    force_reload=FORCE_RELOAD
)

---
## Note Finali

### Troubleshooting
- **Rate limit**: Disconnetti runtime tra un anno e l'altro
- **Errori di rete**: Riprova più tardi o usa VPN
- **Dati mancanti**: Alcune gare potrebbero non avere dati completi

### Prossimo Step
Scarica `f1_dataset_combined.pkl` e caricalo in `02_F1_Data_Analysis.ipynb`